# Fine-tune CNN (ImageNet) di spektrogram — D2

**Pendekatan B.** MobileNetV2 pretrained ImageNet, log-mel spektrogram diperlakukan sebagai
gambar 3-channel, `trainable=True`. Dua fase: warm-up head → unfreeze backbone (LR kecil).

- **Dirancang jalan di Kaggle GPU** (run penuh) **dan** CPU lokal (`SMOKE=True`, subset kecil,
  untuk uji bebas-bug).
- **Test set identik** dengan lokal (240 file) — split diambil dari CSV, bukan diregenerasi.
- Pembanding: frozen YAMNet meanstd = **0.7983** (D2 test).

> Deteksi environment otomatis: kalau ada `/kaggle/input` → mode Kaggle (full);
> selain itu → mode lokal (SMOKE).

In [1]:
import os, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

SEED = 42
tf.keras.utils.set_random_seed(SEED)

KAGGLE = Path("/kaggle/input").exists()
SMOKE = not KAGGLE            # lokal = uji cepat; Kaggle = run penuh
CLASSES = ["ambulance", "firetruck", "police", "traffic"]

# --- audio & fitur (parameter proyek) ---
SR, DURATION = 22050, 3.0
N_SAMPLES = int(SR * DURATION)
N_FFT, HOP, N_MELS = 1024, 512, 64
IMG = 128                      # ukuran input MobileNetV2

# --- lokasi data (auto) ---
if KAGGLE:
    # sesuaikan bila slug berbeda; cari otomatis di /kaggle/input
    inp = Path("/kaggle/input")
    cand = [p for p in inp.rglob("*") if p.is_dir() and p.name == "ambulance"]
    DATA_ROOT = cand[0].parent if cand else inp
    split_csv = next(inp.rglob("split_d2_train.csv"))
    SPLIT_DIR = split_csv.parent
    OUT = Path("/kaggle/working")
else:
    ROOT = Path.cwd().parent.parent if Path.cwd().name == "kaggle" else \
           (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
    DATA_ROOT = ROOT / "Dataset" / "Dataset2"
    SPLIT_DIR = ROOT / "ml"
    OUT = ROOT / "ml" / "artifacts" / "d2_cnn_finetune"
OUT.mkdir(parents=True, exist_ok=True)

print(f"KAGGLE={KAGGLE} · SMOKE={SMOKE}")
print(f"DATA_ROOT = {DATA_ROOT}")
print(f"SPLIT_DIR = {SPLIT_DIR}")
print(f"OUT       = {OUT}")
print(f"GPU       = {tf.config.list_physical_devices('GPU')}")

KAGGLE=False · SMOKE=True
DATA_ROOT = D:\Coding Vscode\Siren Classification\Dataset\Dataset2
SPLIT_DIR = D:\Coding Vscode\Siren Classification\ml
OUT       = D:\Coding Vscode\Siren Classification\ml\artifacts\d2_cnn_finetune
GPU       = []


## 1 · Resolusi path + join ke split

Semua `.wav` ditemukan via `rglob` (label = nama folder induk), lalu di-join ke split CSV
kita lewat **(label, filename)** — filename tidak unik lintas kelas, jadi label wajib ikut.
Ini menjamin train/val/test **persis sama** dengan lokal.

In [2]:
# indeks (label, filename) -> path aktual
wav_index = {}
for p in DATA_ROOT.rglob("*.wav"):
    lab = p.parent.name
    if lab in CLASSES:
        wav_index[(lab, p.name)] = str(p)
print(f"total wav terindeks: {len(wav_index)}")


def load_split(name):
    df = pd.read_csv(SPLIT_DIR / f"split_d2_{name}.csv")
    df["path"] = [wav_index.get((l, f)) for l, f in zip(df.label, df.filename)]
    miss = df.path.isna().sum()
    assert miss == 0, f"{miss} file split '{name}' tak ketemu di DATA_ROOT"
    return df


tr, va, te = load_split("train"), load_split("val"), load_split("test")
print(f"train {len(tr)} · val {len(va)} · test {len(te)}")
assert (len(tr), len(va), len(te)) == (1195, 240, 240) or SMOKE, "ukuran split beda!"

if SMOKE:                      # subset kecil supaya cepat di CPU
    tr = tr.groupby("label").head(12).reset_index(drop=True)
    va = va.groupby("label").head(8).reset_index(drop=True)
    te = te.groupby("label").head(8).reset_index(drop=True)
    print(f"[SMOKE] train {len(tr)} · val {len(va)} · test {len(te)}")

total wav terindeks: 1675
train 1195 · val 240 · test 240
[SMOKE] train 48 · val 32 · test 32


## 2 · Log-mel → citra [0,1]

Tiap klip jadi log-mel `(64,130)`, dinormalisasi per-sampel ke [0,1] (min-max). Resize ke
`128×128` + 3-channel dilakukan **di dalam model** supaya bisa ikut diekspor nanti.

In [3]:
idx = {c: i for i, c in enumerate(CLASSES)}


def logmel(path):
    y, _ = librosa.load(path, sr=SR, mono=True)
    if len(y) < N_SAMPLES:
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    y = y[:N_SAMPLES]
    m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
    m = librosa.power_to_db(m, ref=np.max)                 # dB, ~[-80,0]
    m = (m - m.min()) / (m.max() - m.min() + 1e-8)         # -> [0,1] per sampel
    return m[..., None].astype(np.float32)                 # (64,130,1)


def build_xy(df):
    X = np.stack([logmel(p) for p in df.path])
    y = df.label.map(idx).to_numpy()
    return X, y


t0 = time.time()
Xtr, ytr = build_xy(tr); Xva, yva = build_xy(va); Xte, yte = build_xy(te)
print(f"fitur siap {Xtr.shape} dalam {time.time()-t0:.0f}s")

fitur siap (48, 64, 130, 1) dalam 8s


## 3 · Model — MobileNetV2 di atas spektrogram

Input log-mel `(64,130,1)` → resize `128×128` → 3-channel → skala ke [-1,1]
(format MobileNetV2) → backbone pretrained → GAP → head. Preprocessing masuk ke dalam model
(baik untuk export end-to-end nanti).

In [4]:
def build_model(trainable_base: bool):
    base = tf.keras.applications.MobileNetV2(
        include_top=False, weights="imagenet", input_shape=(IMG, IMG, 3), pooling="avg")
    base.trainable = trainable_base

    inp = tf.keras.Input(shape=(N_MELS, 130, 1))
    x = tf.keras.layers.Resizing(IMG, IMG)(inp)
    x = tf.keras.layers.Concatenate()([x, x, x])           # 1ch -> 3ch
    x = tf.keras.layers.Rescaling(2.0, offset=-1.0)(x)     # [0,1] -> [-1,1]
    x = base(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(len(CLASSES), activation="softmax")(x)
    return tf.keras.Model(inp, out), base


class ValMacroF1(tf.keras.callbacks.Callback):
    def __init__(self, Xv, yv): super().__init__(); self.Xv, self.yv = Xv, yv
    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        p = self.model.predict(self.Xv, verbose=0).argmax(1)
        logs["val_macro_f1"] = f1_score(self.yv, p, average="macro")


def cbs():
    return [ValMacroF1(Xva, yva),
            tf.keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max",
                                             patience=6 if SMOKE else 12,
                                             restore_best_weights=True)]

## 4 · Two-phase training

Fase 1: backbone beku, latih head. Fase 2: buka backbone, LR kecil.

In [5]:
E1, E2 = (2, 2) if SMOKE else (10, 40)

# fase 1
model, base = build_model(trainable_base=False)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("=== Fase 1: head warm-up ===")
model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=E1,
          batch_size=32, callbacks=cbs(), verbose=2)

# fase 2: unfreeze backbone, LR kecil
base.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("\n=== Fase 2: fine-tune backbone ===")
hist = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=E2,
                 batch_size=32, callbacks=cbs(), verbose=2)

      0/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

  49152/9406464 ━━━━━━━━━━━━━━━━━━━━ 40s 4us/step

  81920/9406464 ━━━━━━━━━━━━━━━━━━━━ 45s 5us/step

 131072/9406464 ━━━━━━━━━━━━━━━━━━━━ 33s 4us/step

 163840/9406464 ━━━━━━━━━━━━━━━━━━━━ 30s 3us/step

...


=== Fase 2: fine-tune backbone ===


Epoch 1/2


2/2 - 51s - 26s/step - accuracy: 0.3125 - loss: 1.7858 - val_accuracy: 0.2188 - val_loss: 1.6515 - val_macro_f1: 0.1886


Epoch 2/2


2/2 - 2s - 1s/step - accuracy: 0.8333 - loss: 0.4854 - val_accuracy: 0.2812 - val_loss: 1.6394 - val_macro_f1: 0.2436


## 5 · Evaluasi di test + simpan

In [6]:
y_pred = model.predict(Xte, verbose=0).argmax(1)
macro_f1 = f1_score(yte, y_pred, average="macro")
acc = accuracy_score(yte, y_pred)
print(f"TEST macro-F1 : {macro_f1:.4f}   (frozen meanstd 0.7983 · target 0.85)")
print(f"TEST accuracy : {acc:.4f}\n")
print(classification_report(yte, y_pred, target_names=CLASSES, digits=3))
print("confusion:\n", confusion_matrix(yte, y_pred))

model.save(OUT / "model.keras")
json.dump({
    "exp_id": "d2_cnn_finetune", "backbone": "mobilenetv2_imagenet",
    "feature": "logmel_64x130", "test_macro_f1": float(macro_f1),
    "test_accuracy": float(acc), "smoke": SMOKE,
    "per_class_f1": dict(zip(CLASSES, f1_score(yte, y_pred, average=None).tolist())),
}, open(OUT / "metrics.json", "w"), indent=2)
print(f"\ntersimpan -> {OUT}")
if SMOKE:
    print("\n[SMOKE] angka tidak bermakna (subset kecil, 2+2 epoch) — hanya uji bebas-bug.")
    print("Jalankan di Kaggle GPU (KAGGLE terdeteksi -> SMOKE=False) untuk hasil sebenarnya.")

TEST macro-F1 : 0.1905   (frozen meanstd 0.7983 · target 0.85)
TEST accuracy : 0.2188

              precision    recall  f1-score   support

   ambulance      0.154     0.250     0.190         8
   firetruck      0.333     0.250     0.286         8
      police      0.000     0.000     0.000         8
     traffic      0.231     0.375     0.286         8

    accuracy                          0.219        32
   macro avg      0.179     0.219     0.190        32
weighted avg      0.179     0.219     0.190        32

confusion:
 [[2 3 0 3]
 [3 2 0 3]
 [3 1 0 4]
 [5 0 0 3]]


D:\Coding Vscode\Siren Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\Coding Vscode\Siren Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
D:\Coding Vscode\Siren Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i


tersimpan -> D:\Coding Vscode\Siren Classification\ml\artifacts\d2_cnn_finetune

[SMOKE] angka tidak bermakna (subset kecil, 2+2 epoch) — hanya uji bebas-bug.
Jalankan di Kaggle GPU (KAGGLE terdeteksi -> SMOKE=False) untuk hasil sebenarnya.


---
Setelah run penuh di Kaggle, catat `test_macro_f1` dari `metrics.json` dan bandingkan dengan
notebook YAMNet fine-tune serta frozen meanstd (0.7983). Model pemenang diekspor end-to-end
untuk deployment (Fase 4).